# Feature Extraction

Data representation plays a critical role in the performance of many machine learning methods in machine learning. The data representation of network traffic often determines the effectiveness of these models as much as the model itself. The wide range of novel events that network operators need to detect (e.g., attacks, malware, new applications, changes in traffic demands) introduces the possibility for a broad range of possible models and data representations.

[NetML](https://pypi.org/project/netml/) is an open-source tool and end-to-end pipeline for anomaly detection in network traffic. This notebook walks through the use of that library.

First, let us load the library.

In [1]:
import logging
logging.getLogger("scapy.runtime").setLevel(logging.ERROR)

from netml.pparser.parser import PCAP
from netml.utils.tool import dump_data, load_data

import pandas as pd
import sys
import matplotlib.pyplot as plt

Matplotlib is building the font cache; this may take a moment.


## Specify a Packet Capture File

Create a pcap data structure for which we would like to extract features. You could do this based on the packet capture files that we have been using in previous hands assignments. Any packet capture file will suffice, however.

You can define the minumum number of packets that you want to include in each flow.

In [2]:
netflix_pcap = PCAP('/Users/domi/Desktop/CMSC25422/CMSC-25422/docs/notebooks/data/netflix.pcap')
netflix_pcap_df = PCAP('/Users/domi/Desktop/CMSC25422/CMSC-25422/docs/notebooks/data/netflix.pcap')
netflix_pcap

## Convert the Packet Capture Into Flows

Find the function in `netml` that converts the pcap file into flows. Examing the resulting data structure. What does it contain?

In [3]:
netflix_pcap.pcap2flows()


In [4]:
netflix_pcap_df.pcap2pandas()

In [5]:
ndf = netflix_pcap_df.df
ndf.head()

,datetime,dns_query,dns_resp,ip_dst,ip_dst_int,ip_src,ip_src_int,is_dns,length,mac_dst,mac_dst_int,mac_src,mac_src_int,port_dst,port_src,protocol,time,time_normed
0,2018-02-11 15:10:00,"(fonts.gstatic.com.,)",None,128.93.77.234,2.153598e+09,192.168.43.72,3.232247e+09,True,77,a0:ce:c8:0d:2b:a7,176809980013479,e4:ce:8f:01:4c:54,251575813622868,53.0,55697.0,UDP,1518358200.534682,0.000000
1,2018-02-11 15:10:00,"(fonts.gstatic.com.,)",None,128.93.77.234,2.153598e+09,192.168.43.72,3.232247e+09,True,77,a0:ce:c8:0d:2b:a7,176809980013479,e4:ce:8f:01:4c:54,251575813622868,53.0,59884.0,UDP,1518358200.534832,0.000150
2,2018-02-11 15:10:00,"(googleads.g.doubleclick.net.,)",None,128.93.77.234,2.153598e+09,192.168.43.72,3.232247e+09,True,87,a0:ce:c8:0d:2b:a7,176809980013479,e4:ce:8f:01:4c:54,251575813622868,53.0,61223.0,UDP,1518358200.539408,0.004726
3,2018-02-11 15:10:00,"(googleads.g.doubleclick.net.,)",None,128.93.77.234,2.153598e+09,192.168.43.72,3.232247e+09,True,87,a0:ce:c8:0d:2b:a7,176809980013479,e4:ce:8f:01:4c:54,251575813622868,53.0,58785.0,UDP,1518358200.541204,0.006522
4,2018-02-11 15:10:00,"(ytimg.l.google.com.,)",None,128.93.77.234,2.153598e+09,192.168.43.72,3.232247e+09,True,78,a0:ce:c8:0d:2b:a7,176809980013479,e4:ce:8f:01:4c:54,251575813622868,53.0,51938.0,UDP,1518358200.545785,0.011103


In [6]:
netflix_pcap.pcap2flows()

In [7]:
print(len(netflix_pcap.flows))          # number of flows

fid, packets = netflix_pcap.flows[0]    # first flow
print(fid)                       # (src_ip, dst_ip, src_port, dst_port, proto)
print(len(packets))              # number of packets in that flow
print(packets[0].summary())      # Scapy's built-in packet summary

184
('192.168.43.72', '172.217.18.195', 58443, 443, 6)
12
Ether / IP / TCP 192.168.43.72:58443 > 172.217.18.195:https S


From what I understand, the flows object is a list of tuples (A,B), where A is the identifier of the flow, and B is a list of the packets contained in the flow of communication.

## Explore the Flows

How many flows are in your data structure?

184

What other information does the flow data structure contain, for each flow?

Source IP, Destination IP, Source Port, Destination Port, Protocol

## Extract Features from Each Flow

Use the `netml` library to extract features from each flow. 

The [documentation](https://pypi.org/project/netml/) and [accompanying paper](https://arxiv.org/pdf/2006.16993.pdf) provide examples of features that you can try to extract. 

First try to extract the inter-arrival times for each flow.

### Interarrival Times

In [8]:
# turn flows into a feature matrix, e.g. inter-arrival time
netflix_pcap.flow2features('IAT')
X = netflix_pcap.features   # numpy array — one row per flow, feature vector per row

### Explore the Per-Flow Features

Inspect and print the features for each flow. (If you feel compelled: Get fancy! Plot distributions, etc. Whatever you like!)

In [10]:

rows = []
for fid, packets in netflix_pcap.flows:
    src_ip, dst_ip, src_port, dst_port, proto = fid
    
    timestamps = [float(pkt.time) for pkt in packets]
    byte_sizes = [len(pkt) for pkt in packets]
    
    duration = max(timestamps) - min(timestamps) if len(timestamps) > 1 else 0
    
    rows.append({
        'src_ip': src_ip,
        'dst_ip': dst_ip,
        'src_port': src_port,
        'dst_port': dst_port,
        'protocol': 'TCP' if proto == 6 else 'UDP' if proto == 17 else proto,
        'num_packets': len(packets),
        'total_bytes': sum(byte_sizes),
        'avg_pkt_size': sum(byte_sizes) / len(byte_sizes),
        'duration_sec': duration,
        'start_time': min(timestamps),
    })

flow_summary = pd.DataFrame(rows)
flow_summary.head()

,src_ip,dst_ip,src_port,dst_port,protocol,num_packets,total_bytes,avg_pkt_size,duration_sec,start_time
0,192.168.43.72,172.217.18.195,58443,443,TCP,12,1072,89.333333,11.461576,1.518358e+09
1,192.168.43.72,216.58.209.228,58444,443,TCP,67,5544,82.746269,248.797163,1.518358e+09
2,192.168.43.72,216.58.209.228,58445,443,TCP,5,342,68.400000,19.349723,1.518358e+09
3,192.168.43.72,172.217.18.195,58446,443,TCP,5,342,68.400000,19.349462,1.518358e+09
4,192.168.43.72,172.217.18.195,58447,443,TCP,29,2624,90.482759,300.745291,1.518358e+09


### Other Features and Options

1. Try some of the other features in the `netml` library.

  Here are some of the other possibilities, which can be passed to the library:
  * IAT: A flow is represented as a timeseries of inter-arrival times between packets, i.e., elapsed time in seconds between any two packets in the flow.   
  *  STATS: A flow is represented as a set of statistical quantities. We choose ten of the most common such
statistics in the literature: flow duration, number of packets sent per second, number of bytes
per second, and various statistics on packet sizes within each flow: mean, standard deviation, inter-quartile range,
minimum, and maximum.
  * SIZE: A flow is represented as a timeseries of packet sizes in bytes, with one sample per packet. 
  * SAMP-NUM: A flow is partitioned into small intervals of equal length 𝛿𝑡, and the number of packets in each interval is recorded; thus, a flow is represented as a timeseries of packet counts in small time intervals, with one sample per time interval. Here, 𝛿𝑡 might be viewed as a choice of sampling rate for the timeseries, hence the nomenclature.
  * SAMP-SIZE: A flow is partitioned into time intervals of equal length 𝛿𝑡, and the total packet size (i.e., byte count) in each interval is recorded; thus, a flow is represented as a timeseries of byte counts in small time intervals, with one sample per time interval.
  

  
2. One of the challenges with providing packet traces to models involve ensuring that all feature vectors are of the same length. The `netml` libary will do that for you, but there are a number of different ways to solve the problem. What do some of the following options do?  Explore how different settings of the following affect the dimensionality of your resulting feature vector.

 * flow_ptks_thres
 * q_interval

## Thought Questions

What other features might you want to extract from packet captures that are not provided by the `netml` library?